# 🌐 OmniBrowse Cloud Bridge (Mobile Edition)
### Control OmniBrowse from your Phone using ChatGPT, Claude, or DeepSeek

**Instructions:**
1. Tap the **Play (▶)** button on the setup cells below.
2. Copy the generated **Public URL**.
3. Paste into your Custom GPT / Claude / Phone Browser to control your cloud browser on the go!

In [ ]:
# Step 1: Install Playwright & Fast Cloud Tunnel
!pip install playwright fastapi uvicorn pyngrok nest_asyncio
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# Step 2: Clone your OmniBrowse Repository
!git clone https://github.com/aniketkhalko15-blip/omnibrowse.git /content/omnibrowse
%cd /content/omnibrowse

In [ ]:
# Step 3: Launch Universal Cloud Bridge API Server
import asyncio
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok
import uvicorn
from omnibrowse.browser import OmniBrowser

nest_asyncio.apply()
app = FastAPI(title="OmniBrowse Mobile Cloud Bridge", version="1.0")
browser = None

class NavigateRequest(BaseModel):
    url: str

class ActionRequest(BaseModel):
    element_id: str
    text: str = None

@app.on_event("startup")
async def startup():
    global browser
    browser = OmniBrowser(headless=True)
    await browser.start()

@app.post("/navigate")
async def navigate(req: NavigateRequest):
    """Navigate to URL and return clean AI-ready text view of the page"""
    state = await browser.goto(req.url)
    return state

@app.post("/click")
async def click(req: ActionRequest):
    """Click an indexed element by ID"""
    return await browser.click(req.element_id)

@app.post("/type")
async def type_text(req: ActionRequest):
    """Type text into an indexed input"""
    return await browser.type_text(req.element_id, req.text or "", press_enter=True)

@app.get("/state")
async def get_state():
    """Get current page text & indexed elements for LLM"""
    return await browser.get_state()

# Expose via free tunnel and run
public_url = ngrok.connect(8000).public_url
print("\n" + "="*65)
print(f"🚀 OMNIBROWSE CLOUD BRIDGE IS LIVE!")
print(f"📱 YOUR PUBLIC API URL: {public_url}")
print(f"📄 OPENAPI SCHEMA FOR CHATGPT/CLAUDE: {public_url}/openapi.json")
print("="*65)

uvicorn.run(app, host="0.0.0.0", port=8000)